# Derive and check a neural-network gradient

Prerequisites: Matrix multiplication, derivatives, and the chain rule.

Let h=tanh(Wx), prediction=vᵀh, and loss=½(prediction−target)². Reverse-mode differentiation carries a sensitivity from the scalar loss back through each intermediate. The hidden sensitivity is (prediction−target)v⊙(1−h²); multiplying it by xᵀ gives the gradient of W. Array shape is part of the derivation: a two-entry sensitivity times a two-entry row produces a 2×2 gradient.

The companion implements these reverse-mode equations directly. It independently perturbs each weight by ±ε and estimates a central finite difference. Agreement is a useful implementation check because the numerical path does not reuse the gradient formula. It is not a proof for every network or nondifferentiable point. Very large ε creates truncation error; extremely small ε amplifies rounding.

A small step opposite the gradient should reduce this smooth example's loss. This does not make every learning rate safe or guarantee convergence of a general deep network. PyTorch's autograd automates graph bookkeeping; understanding the hand calculation helps diagnose detached tensors, incorrect reductions and unexpected gradient shapes.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Forward | x → Wx → tanh → vᵀh → loss | Store intermediates |
| Reverse | loss sensitivity → hidden sensitivity → outer product | Apply the chain rule |
| Independent check | Two loss evaluations per weight | Central finite difference |

## Predict

Use x=(0.3,−0.7), W=[[0.2,−0.4],[0.5,0.1]], v=(0.8,−0.2), target 0.4. With ε=10⁻⁵, the maximum absolute gradient discrepancy should be below 10⁻⁸.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import ai_models as m

In [ ]:
result=m.autodiff()
assert result['maximum_error']<1e-8
assert result['loss_after_step']<result['loss']
print(result)
print('Larger epsilon:',m.autodiff(.01))

## Perturb and explain

Write out one gradient entry by hand. Sweep ε through 10⁻², 10⁻⁴, 10⁻⁶ and 10⁻⁸, and explain why a smaller ε is not always more accurate.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

Why compare with finite differences?

1. They always train faster
2. They independently check the implemented derivative
3. They remove floating-point error

Answer and explanations: 1: They require many forward evaluations. 2: The check follows a different computational path. 3: Finite differences themselves have rounding and truncation error.

## Primary evidence

- [PyTorch — Learning with examples, including manual gradients and autograd](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.